# 5주차 — 피드백과 대수 루프

**이번 주에 완성되는 엔진 조각**
```
Diagram compile 에 feedthrough 규칙 추가 — 피드백(고리!)이 가능해진다
```
제어의 본질은 피드백인데 지난주 엔진은 고리를 못 다룬다. 오늘 고친다.

## 이론 요약
- `Gain` 끼리의 고리: y=2x 이면서 x=0.4y — **답이 없는 연립** = 대수 루프
- 상태 블록(적분기)의 출력은 '과거의 누적' — 지금 입력이 필요 없다 →
  **순서 제약에서 빠진다** → 고리가 끊긴다
- 구현은 단 한 줄: `if dst.feedthrough: deps[dst].add(src)` — 이 한 줄이
  Simulink 의 폐루프를 가능하게 하는 전부다

In [ ]:
import importlib, myblocks, mydiagram
importlib.reload(myblocks); importlib.reload(mydiagram)
# 폐루프: ref → err → ctrl → plant(StateStub) ─┐
#          ▲──────────────────────────────────┘
bd = mydiagram.Diagram()
ref = bd.add(myblocks.Constant(1.0, name='ref'))
err = bd.add(myblocks.Sum('+-', name='err'))
ctrl = bd.add(myblocks.Gain(3.0, name='ctrl'))
plant = bd.add(myblocks.StateStub(name='plant'))
bd.connect(ref, err[0]); bd.connect(plant, err[1])
bd.connect(err, ctrl); bd.connect(ctrl, plant)
print('고리가 있는데도 컴파일 성공:', [b.name for b in bd.compile()])

## 과제
채점기의 모델 5개(순수 게인 고리/상태 블록 고리/혼합/긴 고리/무고리)가
전부 올바르게 판별되어야 한다. 대수 루프 에러 메시지에는 고리 블록
이름들을 넣어라 — 미래의 너(디버깅하는 학생)를 위한 배려다.

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week05.py -q